In [1]:
from sklearn.preprocessing import LabelEncoder
from hmmlearn import hmm,vhmm
import pickle
import os
from tqdm import tqdm
os.environ['OMP_NUM_THREADS'] = '10'  # 例如，设置为4个线程

import numpy as np
import seaborn as sns
import pandas as pd
from matplotlib import pyplot as plt
from selenium.webdriver.support.expected_conditions import none_of

plt.rcParams['font.family']='Times New Roman,Microsoft YaHei'# 设置字体族，中文为微软雅黑，英文为Times New Roman
plt.rcParams['mathtext.fontset'] = 'stix' # 设置数%matplotlib qt学公式字体为stix
plt.style.use('seaborn-v0_8-paper')
# 设置全局参数
plt.rcParams['figure.facecolor'] = 'white'  # 设置图形的背景为透明
plt.rcParams['axes.facecolor'] = 'white'    # 设置轴域的背景为透明
plt.rcParams['savefig.facecolor'] = 'white' # 保存图像时背景透明
import matplotlib
# matplotlib.use('TkAgg')
%matplotlib inline
from sklearn.preprocessing import StandardScaler

In [2]:
dfmusic = pd.read_pickle("df_chordkey,pkl")
dfbird = pd.read_pickle('final_birds_df.pkl')
dfbasic = pd.read_pickle("soundfeature_other.pkl")
# 去除重复列
dfbasic = dfbasic.loc[:, ~dfbasic.columns.duplicated()]


# 定义高度替换规则
def replace_level(row):
    if row['place'] == 'ZS':
        return '3 m' if row['level'] == 1 else f"{row['level']}m"
    elif row['place'] == 'JH':
        return {1: '1.5 m', 2: '4 m', 3: '8 m', 4: '14 m'}.get(row['level'], f"{row['level']}m")
    elif row['place'] == 'CM':
        return {1: '1.5 m', 2: '10 m', 3: '16 m', 4: '22 m'}.get(row['level'], f"{row['level']}m")
    return f"{row['level']}m"


# 替换 level 列的值
dfbasic['level'] = dfbasic.apply(replace_level, axis=1)
df = pd.merge(dfmusic, dfbird, on=['Date', 'place', 'level'], how='inner')
df = pd.merge(df, dfbasic, on=['Date', 'place', 'level'], how='inner')
# 栖息地分类映射
habitat_mapping = {
    'Aquatic and Wetland Surface Birds': ['Eurasian Wigeon', 'Tundra Swan', 'Mallard',
                                          'Greater White-fronted Goose', 'Common Goldeneye',
                                          'Green-winged Teal', 'Eurasian Coot', 'Eurasian Moorhen'],

    'Shoreline and Marsh Birds': ['Eurasian Curlew', 'Whimbrel'],

    'Grassland Ground Birds': ['Olive-backed Pipit', 'Yellow-browed Bunting', 'Yellow-billed Grosbeak'],

    'Shrub Layer Birds': ['Light-vented Bulbul', 'Chinese Hwamei', 'Japanese Tit',
                          'Silver-throated Tit', 'Chinese Blackbird', 'Pale-legged Leaf Warbler'],

    'Lower Canopy Birds': ['Pale Thrush', 'Oriental Magpie'],

    'Upper Canopy and Aerial Birds': ["Swinhoe's White-eye"]
}

# 将所有物种映射到栖息地类别
species_to_habitat = {}
for habitat, species_list in habitat_mapping.items():
    for species in species_list:
        species_to_habitat[species] = habitat


# 分类函数
def classify_bird_habitat(species):
    return species_to_habitat.get(species, 'Other')  # 若未匹配到则归类为 'Other'


# 应用分类函数
df['habitat'] = df['bird'].copy().apply(classify_bird_habitat)

# 根据分类映射顺序创建分类类型
habitat_order = list(habitat_mapping.keys())
df['habitat'] = pd.Categorical(df['habitat'], categories=habitat_order, ordered=True)
# 定义和弦质量的映射
chord_quality_map = {
    'maj': '',  # major 不需要后缀
    'min': 'm',  # minor 转换为 'm'
    'dim': 'dim',  # diminished 保持不变
    'aug': 'aug',  # augmented 保持不变
    '7': '7',  # dominant 7th
    'maj7': 'maj7',  # major 7th
    'min7': 'm7',  # minor 7th
    'dim7': 'dim7',  # diminished 7th
    'sus4': 'sus4',  # suspended 4th
    'sus2': 'sus2'  # suspended 2nd
}


def convert_chord_name(chord_name):
    if ':' in chord_name:
        root, quality = chord_name.split(':')
        # 使用映射字典转换和弦质量
        return f"{root}{chord_quality_map.get(quality, '')}"
    else:
        return chord_name


# 应用转换函数
df['event'] = df['event'].apply(convert_chord_name)

dfmusic2 = pd.read_pickle("music_var_chordkey2.pkl")
dfbasic2 = pd.read_pickle("soundfeature_other2.pkl")
# 去除重复列
dfbasic2 = dfbasic2.loc[:, ~dfbasic2.columns.duplicated()]
dfmusic2 = dfmusic2.loc[:, ~dfmusic2.columns.duplicated()]
df['id'] = df.index
df2 = pd.merge(dfmusic2, dfbasic2, on=['file'], how='left', suffixes=('', '_drop'))
df2 = df2[[col for col in df2.columns if not col.endswith('_drop')]]

# 设置24个调性名称
key_names = ['A major', 'Bb major', 'B major', 'C major', 'Db major',
             'D major', 'Eb major', 'E major', 'F major', 'F# major',
             'G major', 'Ab major', 'A minor', 'Bb minor', 'B minor',
             'C minor', 'C# minor', 'D minor', 'D# minor', 'E minor',
             'F minor', 'F# minor', 'G minor', 'G# minor']


# 转换 `key` 列
def parse_key(key_str):
    # 去掉大括号和空格，按逗号分割字符串
    key_str = key_str.strip('{}').replace(' ', '').split(',')
    # 将所有值转换为浮点数
    return np.array([float(k) for k in key_str])


# 应用转换函数
df2['Key'] = df2['Key'].apply(parse_key)
# 从每行的 Key 数组中选择置信度最大的 Key
df2['track'] = df2['Key'].apply(lambda x: key_names[np.argmax(x)])

# 栖息地分类映射
habitat_mapping2 = {
    'Aquatic and Wetland Surface Birds': ['Eurasian Wigeon', 'Tundra Swan', 'Mallard',
                                          'Greater White-fronted Goose', 'Common Goldeneye',
                                          'Green-winged Teal', 'Eurasian Coot', 'Eurasian Moorhen'],

    'Shoreline and Marsh Birds': ['Eurasian Curlew', 'Whimbrel'],

    'Grassland Ground Birds': ['Olive-backed Pipit', 'Yellow-browed Bunting', 'Yellow-billed Grosbeak'],

    'Shrub Layer Birds': ['Light-vented Bulbul', 'Chinese Hwamei', 'Japanese Tit',
                          'Silver-throated Tit', 'Chinese Blackbird', 'Pale-legged Leaf Warbler'],

    'Lower Canopy Birds': ['Pale Thrush', 'Oriental Magpie'],

    'Upper Canopy and Aerial Birds': ["Swinhoe's White-eye"]
}

# 将所有物种映射到栖息地类别
species_to_habitat2 = {}
for habitat, species_list in habitat_mapping2.items():
    for species in species_list:
        species_to_habitat2[species] = habitat


# 分类函数
def classify_bird_habitat(species):
    return species_to_habitat2.get(species, 'Other')  # 若未匹配到则归类为 'Other'


bird_name_mapping = {
    'Eurasian Teal': 'Eurasian Moorhen',
    'Silver-throated Bushtit': 'Silver-throated Tit',
    'Common Moorhen': 'Eurasian Moorhen',
    'Eurasian Whimbrel': 'Whimbrel'
}

# 先转换 `birdenname`
df2['bird'] = df2['birdenname'].replace(bird_name_mapping)

# 应用分类函数
df2['habitat'] = df2['bird'].copy().apply(classify_bird_habitat)

# 根据分类映射顺序创建分类类型
df2['habitat'] = pd.Categorical(df2['habitat'], categories=habitat_order, ordered=True)

# 解析 chord 变量为列表
df2['chord'] = df2['chord'].str.strip('{}').str.split(',')

df2 = df2.explode('chord').rename(columns={'chord': 'event'}).reset_index().rename(columns={'index': 'id'})

In [4]:
df2

,id,Date,birdenname,birdspgen,file,realfilename,rec,cnt,loc,lat,...,H_pairedShannon,H_gamma,H_GiniSimpson,RAOQ,AGI,ROItotal,ROIcover,track,bird,habitat
0,0,2022-01-16 00:00:00,Eurasian Teal,Anas crecca,XC706499.mp3,XC706499,Irish Wildlife Sounds,Ireland,"Tacumshane, Wexford, County Wexford",52.1853,...,3.116815,342.933459,0.846384,0.004398,1.763521,144.0,3.166691,G# minor,Eurasian Moorhen,Aquatic and Wetland Surface Birds
1,0,2022-01-16 00:00:00,Eurasian Teal,Anas crecca,XC706499.mp3,XC706499,Irish Wildlife Sounds,Ireland,"Tacumshane, Wexford, County Wexford",52.1853,...,3.116815,342.933459,0.846384,0.004398,1.763521,144.0,3.166691,G# minor,Eurasian Moorhen,Aquatic and Wetland Surface Birds
2,1,2023-12-13 10:03:00,Eurasian Teal,Anas crecca,XC872164.mp3,XC872164,Johannes Dag Mayer,Germany,"Suedsee, Laupheim, Biberach, Tübingen, Baden-W...",48.2078,...,5.120108,35582.846346,0.942238,0.677393,1.263552,73.0,3.437754,Bb minor,Eurasian Moorhen,Aquatic and Wetland Surface Birds
3,2,2024-11-04 08:11:00,Eurasian Teal,Anas crecca,XC946313.mp3,XC946313,Sonothèque ADVL,France,"Cabane Vauban (Carolles), Manche, Normandie",48.7443,...,4.060774,4023.734559,0.916002,0.054279,1.307255,128.0,9.873588,G# minor,Eurasian Moorhen,Aquatic and Wetland Surface Birds
4,3,2014-07-13 09:00:00,Eurasian Teal,Anas crecca,XC187090.mp3,XC187090,Albert Lastukhin,Russian Federation,"Novocheboksarsk, gorod Novocheboksarsk, Chuvas...",56.1056,...,5.796961,24701.050359,0.987338,0.325825,1.420083,267.0,5.509801,Bb minor,Eurasian Moorhen,Aquatic and Wetland Surface Birds
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91809,14622,2008-02-17 19:05:00,Greater White-fronted Goose,Anser albifrons,XC280889.mp3,XC280889,Peter Boesman,Belgium,"Uitkerkse polders, West-Vlaanderen",51.2950,...,3.902131,880.761415,0.924076,0.016740,4.589344,89.0,4.272296,F# minor,Greater White-fronted Goose,Aquatic and Wetland Surface Birds
91810,14622,2008-02-17 19:05:00,Greater White-fronted Goose,Anser albifrons,XC280889.mp3,XC280889,Peter Boesman,Belgium,"Uitkerkse polders, West-Vlaanderen",51.2950,...,3.902131,880.761415,0.924076,0.016740,4.589344,89.0,4.272296,F# minor,Greater White-fronted Goose,Aquatic and Wetland Surface Birds
91811,14622,2008-02-17 19:05:00,Greater White-fronted Goose,Anser albifrons,XC280889.mp3,XC280889,Peter Boesman,Belgium,"Uitkerkse polders, West-Vlaanderen",51.2950,...,3.902131,880.761415,0.924076,0.016740,4.589344,89.0,4.272296,F# minor,Greater White-fronted Goose,Aquatic and Wetland Surface Birds
91812,14623,2021-11-23 01:36:00,Greater White-fronted Goose,Anser albifrons,XC718649.mp3,XC718649,Chèvremont Fabian,Belgium,"Fléron, Liège, Wallonie",50.6041,...,4.937674,16106.852922,0.948817,0.279683,1.245774,110.0,1.291541,C minor,Greater White-fronted Goose,Aquatic and Wetland Surface Birds


In [3]:
def removechordkey_month(df_chordkey,chordbound=0.01):
    # 计算每个和弦的出现次数
    event_counts = df_chordkey['event'].value_counts()
    # 计算总数的1%
    event_threshold = len(df_chordkey) * chordbound
    # 过滤掉出现次数低于1%的和弦和track
    filtered_events = event_counts[event_counts > event_threshold].index.tolist()
    # 创建一个过滤后的DataFrame
    filtered_df = df_chordkey[df_chordkey['event'].isin(filtered_events)]
    return filtered_df

In [4]:

# 准备存储转移矩阵的字典
transition_matrices_df = {}
transition_matrices_df2 = {}

In [5]:
# 计算总的迭代次数（总的组合数）
total_combinations = len(habitat_order) 

# 使用 tqdm 进度条包装循环
for habitat in tqdm(habitat_order, desc="Processing places"):
    # 过滤数据
    filtered_df = removechordkey_month(df[df['habitat'] == habitat].copy(),chordbound=0.01)
    # 按 Date 排序
    filtered_df = filtered_df.sort_values(by=['Date', 'id'], ascending=[True, True])
    # 将和弦（event）编码为整数
    le = LabelEncoder()
    encoded_events = le.fit_transform(filtered_df['event'])
    # 准备和弦序列数据
    sequences = encoded_events.reshape(-1, 1)
    lengths = len(sequences)
    
    if lengths > 1:  # 确保有足够的数据来训练 HMM
        # 创建并训练 HMM 模型
        n_states = len(le.classes_)  # 独特的和弦数量作为状态数
       # 定义隐马尔可夫模型，使用MultinomialHMM
        model = vhmm.VariationalCategoricalHMM(
            n_components=n_states,  # 状态数
            n_iter=1000            # 最大迭代次数
        )
    
        model.fit(sequences)
    
        # 获取转移矩阵
        transition_matrix = model.transmat_
    
        # 创建转移矩阵的 DataFrame，并将和弦名称作为索引和列
        transition_matrix_df = pd.DataFrame(
            transition_matrix, 
            index=le.classes_, 
            columns=le.classes_
        )
    
        # 构建字典的键
        key = f'{habitat}'
        transition_matrices_df[key] = transition_matrix_df

Processing places: 100%|██████████| 6/6 [16:59<00:00, 169.87s/it]


In [6]:
# 转移矩阵字典保存为文件（例如，pickle 文件）
with open('transition_matrices_df.pkl', 'wb') as f:
    pickle.dump(transition_matrices_df, f)

In [7]:
# 计算总的迭代次数（总的组合数）
total_combinations = len(habitat_order) 

# 使用 tqdm 进度条包装循环
for habitat in tqdm(habitat_order, desc="Processing places"):
# 过滤数据
    filtered_df = removechordkey_month(df2[df2['habitat'] == habitat].copy(), chordbound=0.01)

    # 先对所有和弦进行编码
    le = LabelEncoder()
    le.fit(filtered_df['event'])

    all_sequences = []  # 存储所有样本的数据
    all_lengths = []  # 存储每个样本的长度

    # 按样本 ID 分组
    for sample_id, group in filtered_df.groupby("id"):
        encoded_events = le.transform(group['event'])  # 进行编码

        if len(encoded_events) == 1:
            # **处理 length=1 的情况**（复制一份或设置自循环）
            encoded_events = np.append(encoded_events, encoded_events[0])

        all_sequences.extend(encoded_events.tolist())
        all_lengths.append(len(encoded_events))

    if len(all_sequences) > 1:  # 确保数据足够训练 HMM
        all_sequences = np.array(all_sequences).reshape(-1, 1)

        n_states = len(le.classes_)

        # 训练 HMM
        model = vhmm.VariationalCategoricalHMM(
            n_components=n_states,
            n_iter=1000
        )

        model.fit(all_sequences, lengths=all_lengths)  # **关键点：使用 lengths 处理样本边界**

        # 获取转移矩阵
        transition_matrix = model.transmat_

        # 创建转移矩阵的 DataFrame
        transition_matrix_df = pd.DataFrame(
            transition_matrix,
            index=le.classes_,
            columns=le.classes_
        )

        # 存储到结果字典
        transition_matrices_df2[habitat] = transition_matrix_df

Processing places: 100%|██████████| 6/6 [35:02<00:00, 350.46s/it]  


In [9]:
# 转移矩阵字典保存为文件（例如，pickle文件）
with open('transition_matrices_df2_0.01.pkl', 'wb') as f:
    pickle.dump(transition_matrices_df2, f)

In [25]:
# 转移矩阵字典保存为文件（例如，pickle 文件）
with open('transition_matrices_df2.pkl', 'wb') as f:
    pickle.dump(transition_matrices_df2, f)

In [17]:
# 计算总的迭代次数（总的组合数）
total_combinations = len(habitat_order)

# **处理每个 habitat**
for habitat in tqdm(habitat_order, desc="Processing habitats"):
    # 过滤数据
    filtered_df = removechordkey_month(df2[df2['habitat'] == habitat].copy(), chordbound=0.05)

    le = LabelEncoder()
    le.fit(filtered_df['event'])  # 对所有和弦进行编码

    all_sequences = []  # 存储所有拼接的和弦序列
    all_lengths = []  # 记录每个样本的长度
    pad_symbol = len(le.classes_)  # **定义终止状态（PAD）**

    # **按 id（样本）分组**
    for sample_id, group in filtered_df.groupby("id"):
        encoded_events = le.transform(group['event'])

        if len(encoded_events) == 1:
            # **如果样本只有一个和弦，复制一份**
            encoded_events = np.append(encoded_events, encoded_events[0])

        # **在每个样本后插入终止状态（PAD）**
        encoded_events = np.append(encoded_events, pad_symbol)

        all_sequences.extend(encoded_events.tolist())
        all_lengths.append(len(encoded_events))

    if len(all_sequences) > 1:
        all_sequences = np.array(all_sequences).reshape(-1, 1)

        n_states = len(le.classes_) + 1  # **状态数 +1（包含终止状态 PAD）**

        # **训练 HMM**
        model = vhmm.VariationalCategoricalHMM(
            n_components=n_states,
            n_iter=1000
        )
        model.fit(all_sequences, lengths=all_lengths)  # **使用 lengths 确保样本间不会学习错误的转移**

        # **获取转移矩阵**
        transition_matrix = model.transmat_

        # **创建转移矩阵的 DataFrame**
        transition_matrix_df = pd.DataFrame(
            transition_matrix[:-1, :-1],  # **去掉终止状态（PAD）对应的行列**
            index=le.classes_,
            columns=le.classes_
        )

        # **存储结果**
        transition_matrices_df2[habitat] = transition_matrix_df

Processing habitats: 100%|██████████| 6/6 [03:45<00:00, 37.58s/it] 


In [21]:
# 转移矩阵字典保存为文件（例如，pickle 文件）
with open('transition_matrices_df2.pkl', 'wb') as f:
    pickle.dump(transition_matrices_df2, f)